# Customer Lifetime Value, RFM & Cohort Retention Analytics
Recruiter-ready analysis notebook using the project CSV data.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\norders=pd.read_csv('../data/raw/orders.csv', parse_dates=['order_date'])\ncustomers=pd.read_csv('../data/raw/customers.csv', parse_dates=['signup_date'])\nproducts=pd.read_csv('../data/raw/products.csv')\norders.head()

## 1. RFM Scoring

In [ ]:
snapshot=orders.order_date.max()+pd.Timedelta(days=1)\nrfm=orders.groupby('customer_id').agg(\n recency_days=('order_date', lambda x:(snapshot-x.max()).days),\n frequency=('order_id','nunique'),\n monetary=('net_sales','sum')\n).reset_index()\nrfm['R_score']=pd.qcut(rfm['recency_days'].rank(method='first'),5,labels=[5,4,3,2,1]).astype(int)\nrfm['F_score']=pd.qcut(rfm['frequency'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)\nrfm['M_score']=pd.qcut(rfm['monetary'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)\nrfm['rfm_score']=rfm[['R_score','F_score','M_score']].astype(str).agg(''.join,axis=1)\nrfm.head()

## 2. CLV Estimation**Simplified portfolio CLV:** AOV × annual purchase frequency × expected lifetime years × gross margin factor. This is a planning estimate, not a causal prediction model.

In [ ]:
cust=orders.groupby('customer_id').agg(first_purchase=('order_date','min'),last_purchase=('order_date','max'),orders=('order_id','nunique'),revenue=('net_sales','sum'),profit=('gross_profit','sum')).reset_index()\ncust['tenure_years']=((cust['last_purchase']-cust['first_purchase']).dt.days.clip(lower=30)/365)\ncust['aov']=cust['revenue']/cust['orders']\ncust['annual_frequency']=cust['orders']/cust['tenure_years'].clip(lower=0.25)\ncust['gross_margin']=cust['profit']/cust['revenue']\ncust['estimated_clv']=cust['aov']*cust['annual_frequency']*2.2*cust['gross_margin'].clip(lower=0.20)\ncust[['customer_id','estimated_clv']].sort_values('estimated_clv',ascending=False).head(10)

## 3. Cohort Retention Heatmap

In [ ]:
o=orders.copy()\no['order_month']=o.order_date.dt.to_period('M')\no['cohort_month']=o.groupby('customer_id').order_date.transform('min').dt.to_period('M')\ncohort=o.groupby(['cohort_month','order_month']).customer_id.nunique().reset_index()\ncohort['age']=cohort.order_month.astype(int)-cohort.cohort_month.astype(int)\npivot=cohort.pivot(index='cohort_month',columns='age',values='customer_id')\npct=pivot.div(pivot[0],axis=0)*100\nplt.figure(figsize=(12,7)); sns.heatmap(pct,annot=True,fmt='.0f',cmap='Blues'); plt.title('Customer Cohort Retention (%)'); plt.xlabel('Months Since First Purchase'); plt.ylabel('Cohort Month'); plt.tight_layout(); plt.show()

## 4. Business Recommendations
- Prioritize Champions/Loyal Customers for retention and cross-sell campaigns.\n- Trigger win-back campaigns for high-value At Risk customers.\n- Use category-level demand trends to set reorder points and safety stock.\n- Align paid acquisition with channels that generate high CLV, not just first-order revenue.